# Setup From GitHub

Set `REPO_URL` to your GitHub repo. These notebooks avoid reinstalling PyTorch because Kaggle already ships a CUDA-compatible build.

In [ ]:
REPO_URL = "https://github.com/ahmad903501/qwen-dpo-alignment.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/qwen-dpo-alignment"


In [ ]:
from pathlib import Path

repo_path = Path(REPO_DIR)
if not repo_path.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!pwd


In [ ]:
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps

import torch, transformers, datasets
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
!nvidia-smi

import os
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception as exc:
    print(f"No Kaggle HF_TOKEN secret loaded: {exc}")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])



# Training Only

This notebook trains SFT and DPO checkpoints. It does not generate evaluation outputs or run judges. After it finishes, save the Kaggle notebook version and reuse `outputs/` from another notebook as an attached input dataset.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
MODEL_RUN_NAME = MODEL_NAME.split("/")[-1].lower().replace(".", "-")

# Student-ready small-model run. Use smaller values for smoke tests.
SFT_TRAIN_EXAMPLES = 30000
SFT_VAL_EXAMPLES = 1000
DPO_TRAIN_EXAMPLES = 15000
DPO_VAL_EXAMPLES = 1000

# Use Anthropic/hh-rlhf for HH/safety preference tuning, or
# HuggingFaceH4/ultrafeedback_binarized for broader helpfulness preference tuning.
DPO_DATASET_NAME = "HuggingFaceH4/ultrafeedback_binarized"
DPO_DATASET_TAG = "ultrafeedback"

SFT_DIR = f"outputs/sft-{MODEL_RUN_NAME}"
DPO_DIR = f"outputs/dpo-{MODEL_RUN_NAME}-{DPO_DATASET_TAG}"
PLOT_PREFIX = f"{MODEL_RUN_NAME}_sft_dpo"

## SFT

In [ ]:
!python scripts/train_sft.py \
  --model_name_or_path {MODEL_NAME} \
  --output_dir {SFT_DIR} \
  --max_train_examples {SFT_TRAIN_EXAMPLES} \
  --max_val_examples {SFT_VAL_EXAMPLES} \
  --per_device_batch_size 2 \
  --gradient_accumulation_steps 16 \
  --eval_steps 250


## DPO

Uses `cuda:0` for the policy and `cuda:1` for the frozen reference when two T4s are available.

In [ ]:
!python scripts/train_dpo.py \
  --model_name_or_path {SFT_DIR}/best \
  --dataset_name {DPO_DATASET_NAME} \
  --output_dir {DPO_DIR} \
  --max_train_examples {DPO_TRAIN_EXAMPLES} \
  --max_val_examples {DPO_VAL_EXAMPLES} \
  --per_device_batch_size 1 \
  --gradient_accumulation_steps 32 \
  --eval_steps 25 \
  --beta 0.1 \
  --learning_rate 1e-6


## Check Artifacts

In [ ]:
!find outputs -maxdepth 4 -type f | sort | sed -n '1,240p'


## Plot Training Curves

After SFT and DPO finish, this writes one PNG per logged metric under `outputs/plots`.


In [ ]:
!python scripts/plot_training_curves.py \
  --metrics_path {SFT_DIR}/metrics.jsonl \
  --metrics_path {DPO_DIR}/metrics.jsonl \
  --output_dir outputs/plots \
  --prefix {PLOT_PREFIX}
